# Notebook 06 -- Equilibrium Laplacian RG (Figure 16)

This notebook implements the entropy-based equilibrium variant of Laplacian
renormalization and applies it to three networks.

**Figure produced:**
- **Figure 16:** 1x3 panel -- equilibrium Laplacian RG compression curves
  for NetSci Collaboration, C. Elegans, and Tortoise networks.

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import scipy.linalg
import networkx as nx
import matplotlib.pyplot as plt

from harmonic_morphisms import H_CF_cluster
from harmonic_morphisms.simplicial import import_network_data, compute_entropic_C

DATA_DIR = Path("../data")

SMALL_SIZE = 10; MEDIUM_SIZE = 12; BIGGER_SIZE = 13
plt.rc("font", size=SMALL_SIZE)
plt.rc("axes", titlesize=SMALL_SIZE)
plt.rc("axes", labelsize=MEDIUM_SIZE)
plt.rc("xtick", labelsize=SMALL_SIZE)
plt.rc("ytick", labelsize=SMALL_SIZE)
plt.rc("legend", fontsize=SMALL_SIZE)
plt.rc("figure", titlesize=BIGGER_SIZE)

## Equilibrium Laplacian RG Functions

These functions implement the equilibrium variant, where the diffusion time is
set by the Von Neumann entropy rather than swept linearly. The merging criterion
uses the metric $\rho_{ij}\rho_{ji}/(\rho_{ii}\rho_{jj})$ and greedily merges
pairs until the target number of components is reached.

In [ ]:
def renorm_graph_harmonic_eq(Ag, t_e, se, fraction, Lap, kappa=1):
    """Equilibrium Laplacian renormalization at a given compression fraction."""
    N = len(Ag.nodes())
    target_value = np.log((1 - kappa * fraction) * N)
    t_index = np.argmin(np.abs(se - target_value))
    t = t_e[t_index]

    rho = scipy.linalg.expm(-t * Lap)
    node_list = list(Ag.nodes())
    pair_metrics = []
    for i in range(N):
        for j in range(i + 1, N):
            metric = (rho[i, j] * rho[j, i]) / (rho[i, i] * rho[j, j])
            pair_metrics.append((metric, i, j))
    pair_metrics.sort(reverse=True, key=lambda x: x[0])

    Gv = nx.Graph()
    Gv.add_nodes_from(node_list)
    target_components = int((1 - fraction) * N)
    for metric, i, j in pair_metrics:
        Gv.add_edge(node_list[i], node_list[j])
        if nx.number_connected_components(Gv) <= target_components:
            break

    idx_components = {u: ci for ci, node_set in enumerate(nx.connected_components(Gv)) for u in node_set}
    clusters = {node: idx_components[node] for node in Gv.nodes()}

    G, deg_h, M_deg_h, std_h, av_h, std_v_h, Not_h, \
        deg_cf, M_deg_cf, std_cf, av_cf, std_v_cf, Not_cf = H_CF_cluster(Ag, clusters)
    return G, deg_h, M_deg_h, std_h, av_h, std_v_h, Not_h, \
        deg_cf, M_deg_cf, std_cf, av_cf, std_v_cf, Not_cf, Gv


def H_CF_curves_eq(Ag, t_e, se, n_f, Lap, kappa=1, stability=True):
    """Sweep compression fractions for equilibrium Laplacian RG."""
    g, M_DEG_H, M_DEG_CF, STD_H = [], [], [], []
    interval = np.linspace(0, 1, n_f)
    if stability:
        interval = interval[1:-1]

    for fraction in interval:
        G, _, M_deg_h, std_h, _, _, _, _, M_deg_cf, _, _, _, _, _ = \
            renorm_graph_harmonic_eq(Ag, t_e, se, fraction, Lap, kappa=kappa)
        g.append(G)
        M_DEG_H.append(M_deg_h)
        M_DEG_CF.append(M_deg_cf)
        STD_H.append(std_h)

    return g, M_DEG_H, M_DEG_CF, STD_H, interval

## Load Networks and Run Equilibrium Laplacian RG

In [ ]:
networks = [
    ("out.dimacs10-netscience", "NetSci Collaboration"),
    ("out.dimacs10-celegans_metabolic", "C. Elegans"),
    ("reptilia-tortoise-network-fi.edges", "Tortoise"),
]

results = {}

for filename, display_name in networks:
    print(f"\n{'='*60}")
    print(f"Processing {display_name}")
    print(f"{'='*60}")

    # Load network
    f = open(DATA_DIR / "networks" / filename)
    Ag = import_network_data(f)
    print(f"{display_name}: {Ag.number_of_nodes()} nodes, {Ag.number_of_edges()} edges")

    # Compute Laplacian and eigenvalues
    L0 = nx.laplacian_matrix(Ag, nodelist=Ag.nodes()).todense()
    L0 = np.array(L0, dtype=float)
    e = np.linalg.eigvalsh(L0)
    print(f"Eigenvalue range: [{e[0]:.4f}, {e[-1]:.4f}]")

    # Compute Von Neumann entropy
    _, t_e, se = compute_entropic_C(e, -2, 3, 1000)
    print(f"Entropy range: [{se.min():.4f}, {se.max():.4f}]")

    # Run equilibrium Laplacian RG sweep
    print(f"Running H_CF_curves_eq with 50 fractions...")
    g, M_DEG_H, M_DEG_CF, STD_H, interval = H_CF_curves_eq(
        Ag, t_e, se, 50, L0, stability=False
    )
    print(f"Done. {len(g)} compression steps computed.")

    results[display_name] = {
        "g": g, "M_DEG_H": M_DEG_H, "M_DEG_CF": M_DEG_CF,
        "STD_H": STD_H, "interval": interval,
    }

## Figure 16 -- Equilibrium Laplacian Renormalization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (_, display_name) in zip(axes, networks):
    r = results[display_name]
    interval = r["interval"]
    ax.plot(interval, r["M_DEG_H"], "ob--", linewidth=1.5, markersize=4, label="H Mod.")
    ax.plot(interval, r["M_DEG_CF"], "og--", linewidth=1.5, markersize=4, label="CF Mod.")
    ax.plot(interval, r["STD_H"], "ok--", linewidth=1.5, markersize=4, label="STD H")
    ax.set_xlabel("Compression fraction")
    ax.set_ylabel("Harmonic degree")
    ax.set_ylim([-0.1, 1.2])
    ax.set_title(display_name)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("Equilibrium Laplacian Renormalization")
plt.tight_layout()
plt.savefig("fig16_equilibrium_laplacian.pdf", bbox_inches="tight")
plt.show()
print("Saved fig16_equilibrium_laplacian.pdf")